# Clustering K-Means y Análisis PCA
## Inserción Laboral de Migrantes en Chile

**Objetivo:** Identificar perfiles diferenciados de inserción laboral mediante aprendizaje no supervisado.

**Técnicas aplicadas:**
- **PCA** (Análisis de Componentes Principales): reducción de dimensionalidad y visualización
- **K-Means**: segmentación de perfiles laborales

**Justificación K-Means vs DBSCAN:** Se evalúan ambos algoritmos y se selecciona el más apropiado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.impute import SimpleImputer
import os

BASE = os.path.abspath(os.path.join('..', '..'))
DATA = os.path.join(BASE, 'output', 'data')
FIG  = os.path.join(BASE, 'output', 'fig')
os.makedirs(FIG, exist_ok=True)

sns.set_theme(style='whitegrid', font_scale=1.1)
np.random.seed(42)
print("Librerías cargadas.")

In [ ]:
# Cargar CASEN procesada (tiene todas las variables laborales y de bienestar)
cas = pd.read_parquet(os.path.join(DATA, 'casen_procesada.parquet'))
cas['grupo'] = cas['migrante'].map({0: 'Chilenos', 1: 'Migrantes'})

print(f"CASEN procesada: {cas.shape}")

## 1. Selección y preparación de variables para clustering

In [ ]:
# Variables para clustering: capturan la calidad multidimensional de la inserción
# Solo incluimos personas ocupadas (activ==1) con datos suficientes
cas_ocup = cas[(cas['activ'] == 1) & (cas['migrante'].notna())].copy()
print(f"Ocupados: {len(cas_ocup):,}")

# Variables de clustering
VARS_CLU = [
    'ingreso_hora',       # ingreso por hora trabajada
    'horas_semana',       # horas trabajadas
    'formal',             # formalidad laboral
    'cotiza',             # protección social (cotización)
    'tiene_contrato',     # estabilidad (contrato)
    'dim_adecuacion',     # adecuación ocupacional
    'educa',              # nivel educacional granular (1-7)
    'edad',               # ciclo vital
]
VARS_CLU = [v for v in VARS_CLU if v in cas_ocup.columns]
print(f"Variables de clustering: {VARS_CLU}")

X_clu_raw = cas_ocup[VARS_CLU].copy()
print(f"Datos antes de imputación: {X_clu_raw.shape}")
print("Nulos por variable:")
print(X_clu_raw.isnull().sum().to_string())

In [ ]:
# Imputar valores faltantes con la mediana
imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(
    imputer.fit_transform(X_clu_raw),
    columns=VARS_CLU
)

# Escalar (OBLIGATORIO para K-Means basado en distancias euclidianas)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_imp)

print(f"Dataset clustering (escalado): {X_sc.shape}")
print("Verificación escala (media y std por variable):")
check = pd.DataFrame(X_sc, columns=VARS_CLU)
print(check.describe().loc[['mean','std']].round(3).to_string())

## 2. Análisis PCA: reducción de dimensionalidad

In [ ]:
# PCA completo para entender varianza explicada
pca_full = PCA(random_state=42)
pca_full.fit(X_sc)

varianza_acum = np.cumsum(pca_full.explained_variance_ratio_)
n_componentes_80 = np.searchsorted(varianza_acum, 0.80) + 1
n_componentes_90 = np.searchsorted(varianza_acum, 0.90) + 1

print(f"Componentes para explicar 80% de varianza: {n_componentes_80}")
print(f"Componentes para explicar 90% de varianza: {n_componentes_90}")
print()
print("Varianza explicada por componente:")
for i, (var, acum) in enumerate(zip(pca_full.explained_variance_ratio_, varianza_acum)):
    print(f"  PC{i+1}: {var*100:.2f}%  |  Acumulada: {acum*100:.2f}%")

In [ ]:
# Figura 17: Scree plot y varianza acumulada
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

n_comp = len(pca_full.explained_variance_ratio_)
ax1, ax2 = axes

# Scree plot
ax1.bar(range(1, n_comp+1), pca_full.explained_variance_ratio_ * 100,
        color='#0033A0', alpha=0.8, edgecolor='white')
ax1.set_xlabel("Componente Principal")
ax1.set_ylabel("Varianza Explicada (%)")
ax1.set_title("Scree Plot — Varianza por Componente", fontsize=13, fontweight='bold')

# Varianza acumulada
ax2.plot(range(1, n_comp+1), varianza_acum * 100, 'o-', color='#0033A0', linewidth=2)
ax2.axhline(80, color='orange', linestyle='--', label='80% umbral')
ax2.axhline(90, color='red', linestyle='--', label='90% umbral')
ax2.set_xlabel("Número de Componentes")
ax2.set_ylabel("Varianza Acumulada (%)")
ax2.set_title("Varianza Acumulada", fontsize=13, fontweight='bold')
ax2.legend()

fig.suptitle("Fig. 17: Análisis de Componentes Principales (PCA)", fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig(os.path.join(FIG, 'fig17_pca_varianza.png'))
plt.show()

In [ ]:
# PCA con 2 componentes para visualización
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_sc)

print(f"Varianza explicada por PC1+PC2: {pca2.explained_variance_ratio_.sum()*100:.1f}%")
print()
print("Loadings (contribución de cada variable a cada componente):")
loadings = pd.DataFrame(
    pca2.components_.T,
    columns=['PC1', 'PC2'],
    index=VARS_CLU
)
print(loadings.round(3).to_string())

## 3. K-Means: método del codo y silhouette
**K-Means requiere normalización previa** (ya aplicada) porque minimiza la distancia euclidiana.

In [ ]:
# Método del codo: inercia para K = 2 a 10
inercias = []
silhouettes = []
db_scores = []
K_range = range(2, 11)

print("Calculando K-Means para K=2..10...")
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=500)
    km.fit(X_sc)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_sc, km.labels_, sample_size=10000, random_state=42))
    db_scores.append(davies_bouldin_score(X_sc, km.labels_))
    print(f"  K={k}: Inercia={km.inertia_:,.0f} | Silhouette={silhouettes[-1]:.4f} | Davies-Bouldin={db_scores[-1]:.4f}")

print("\nResumen de métricas:")
metricas_k = pd.DataFrame({
    'K': list(K_range),
    'Inercia': inercias,
    'Silhouette': silhouettes,
    'Davies-Bouldin': db_scores
})
print(metricas_k.to_string(index=False))

In [ ]:
# Figura 18: Método del codo y silhouette
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Inercia (codo)
axes[0].plot(list(K_range), inercias, 'o-', color='#0033A0', linewidth=2, markersize=8)
axes[0].set_xlabel("Número de Clusters (K)")
axes[0].set_ylabel("Inercia (suma de distancias²)")
axes[0].set_title("Método del Codo", fontsize=13, fontweight='bold')
axes[0].set_xticks(list(K_range))

# Silhouette
axes[1].plot(list(K_range), silhouettes, 'o-', color='#27ae60', linewidth=2, markersize=8)
k_optimo = list(K_range)[np.argmax(silhouettes)]
axes[1].axvline(k_optimo, color='red', linestyle='--', label=f'K óptimo = {k_optimo}')
axes[1].set_xlabel("Número de Clusters (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Coeficiente de Silhouette", fontsize=13, fontweight='bold')
axes[1].set_xticks(list(K_range))
axes[1].legend()

fig.suptitle("Fig. 18: Selección del Número de Clusters (K-Means)", fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig(os.path.join(FIG, 'fig18_codo_silhouette.png'))
plt.show()
print(f"K óptimo según Silhouette: {k_optimo}")

## 4. Comparación K-Means vs DBSCAN

In [ ]:
# DBSCAN: alternativa basada en densidad
# Ventaja: no requiere K predefinido, detecta outliers
# Desventaja: sensible a hiperparámetros, difícil con datos de alta dimensión

from sklearn.cluster import DBSCAN

# Usar muestra para eficiencia
idx_sample = np.random.choice(len(X_sc), size=min(20000, len(X_sc)), replace=False)
X_sample = X_sc[idx_sample]

# DBSCAN con parámetros estándar (epsilon basado en escala normalizada)
db = DBSCAN(eps=0.8, min_samples=100, n_jobs=-1)
labels_db = db.fit_predict(X_sample)

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = (labels_db == -1).sum()
pct_noise = n_noise / len(labels_db) * 100

print("=== DBSCAN ===")
print(f"Clusters encontrados: {n_clusters_db}")
print(f"Puntos ruido (outliers): {n_noise:,} ({pct_noise:.1f}%)")
print(f"Distribución por cluster: {pd.Series(labels_db).value_counts().to_dict()}")
print()
if n_clusters_db >= 2:
    mask_valid = labels_db != -1
    sil_db = silhouette_score(X_sample[mask_valid], labels_db[mask_valid], sample_size=5000)
    print(f"Silhouette DBSCAN (sin ruido): {sil_db:.4f}")
else:
    print("DBSCAN generó menos de 2 clusters: no calculable silhouette comparable.")

In [ ]:
justificacion_km = [
    "=== JUSTIFICACION: K-MEANS SOBRE DBSCAN ===",
    "",
    "Se selecciona K-Means como algoritmo principal de clustering por:",
    "",
    "1. DATOS ESTRUCTURADOS Y CONTINUOS:",
    "   Las variables laborales (ingresos, horas, formalidad) forman",
    "   clusters compactos y esfericos, ajuste ideal para K-Means.",
    "   DBSCAN asume clusters de densidad arbitraria, mas util para",
    "   deteccion de anomalias espaciales (ej. Mercado Publico).",
    "",
    "2. PORCENTAJE DE RUIDO EN DBSCAN:",
    "   DBSCAN clasifica un % elevado como ruido (-1), descartando",
    "   observaciones laboralmente relevantes que no son outliers",
    "   reales, sino casos limite entre perfiles.",
    "",
    "3. INTERPRETABILIDAD DE PERFILES:",
    "   K-Means produce centroides claros y comparables entre grupos,",
    "   permitiendo describir perfiles sociologicamente significativos.",
    "   Los clusters DBSCAN son dificiles de caracterizar sistematicamente.",
    "",
    "4. CONSISTENCIA METODOLOGICA:",
    "   El IILM ya captura la multidimensionalidad; el clustering busca",
    "   agrupar trayectorias, no detectar anomalias. K-Means es mas",
    "   apropiado para esta finalidad.",
]
for l in justificacion_km:
    print(l)

## 5. K-Means final y caracterización de perfiles

In [ ]:
# K-Means con K óptimo identificado
K_FINAL = k_optimo

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20, max_iter=1000)
cas_ocup['cluster'] = km_final.fit_predict(X_sc)

print(f"K-Means final con K={K_FINAL}")
print(f"Silhouette: {silhouette_score(X_sc, cas_ocup['cluster'], sample_size=10000, random_state=42):.4f}")
print(f"Inercia: {km_final.inertia_:,.0f}")
print()
print("Distribución por cluster:")
print(cas_ocup['cluster'].value_counts().sort_index().to_string())

In [ ]:
# Caracterizar clusters: centroides en escala original
centroides_sc  = km_final.cluster_centers_
centroides_orig = scaler.inverse_transform(centroides_sc)
df_centroides  = pd.DataFrame(centroides_orig, columns=VARS_CLU)
df_centroides.index = [f"Cluster {i}" for i in range(K_FINAL)]

print("=== CENTROIDES EN ESCALA ORIGINAL ===")
print(df_centroides.round(3).to_string())

In [ ]:
# Composición migratoria de cada cluster
print("\n=== COMPOSICIÓN MIGRATORIA POR CLUSTER ===")
comp_mig = cas_ocup.groupby('cluster')['migrante'].agg(['mean','sum','count'])
comp_mig.columns = ['pct_migrantes','n_migrantes','n_total']
comp_mig['pct_migrantes'] = (comp_mig['pct_migrantes'] * 100).round(2)
comp_mig.index = [f"Cluster {i}" for i in comp_mig.index]
print(comp_mig.to_string())
print()
print(f"Tasa migratoria global en muestra de ocupados: {cas_ocup['migrante'].mean()*100:.2f}%")

In [ ]:
# IILM por cluster
print("\n=== IILM POR CLUSTER ===")
iilm_cluster = cas_ocup.groupby('cluster')['iilm'].agg(['mean','median'])
iilm_cluster.index = [f"Cluster {i}" for i in iilm_cluster.index]
print(iilm_cluster.round(3).to_string())

# Perfil descriptivo
print("\n=== CARACTERIZACIÓN DE PERFILES ===")
perfil_vars = VARS_CLU + ['iilm']
perfil = cas_ocup.groupby('cluster')[perfil_vars].mean().round(3)
perfil.index = [f"Cluster {i}" for i in perfil.index]
print(perfil.to_string())

In [ ]:
# Figura 19: Scatter PCA coloreado por cluster
pca2_coords = pd.DataFrame(X_pca2, columns=['PC1','PC2'])
pca2_coords['cluster'] = cas_ocup['cluster'].values
pca2_coords['migrante'] = cas_ocup['migrante'].values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: coloreado por cluster
colors_clust = sns.color_palette('tab10', K_FINAL)
for k in range(K_FINAL):
    mask = pca2_coords['cluster'] == k
    sample_mask = mask & (np.random.random(len(mask)) < 0.2)
    axes[0].scatter(
        pca2_coords.loc[sample_mask, 'PC1'],
        pca2_coords.loc[sample_mask, 'PC2'],
        c=[colors_clust[k]], alpha=0.4, s=8, label=f'Cluster {k}'
    )
axes[0].set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].set_title("Clusters en Espacio PCA", fontsize=13, fontweight='bold')
axes[0].legend(markerscale=3, fontsize=10)

# Panel 2: coloreado por condición migratoria
colores_mig = {0: '#0033A0', 1: '#fbcb05'}
for m_val, m_label in [(0,'Chileno'), (1,'Migrante')]:
    mask = pca2_coords['migrante'] == m_val
    sample_mask = mask & (np.random.random(len(mask)) < 0.15)
    axes[1].scatter(
        pca2_coords.loc[sample_mask, 'PC1'],
        pca2_coords.loc[sample_mask, 'PC2'],
        c=[colores_mig[m_val]], alpha=0.4, s=8, label=m_label
    )
axes[1].set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)")
axes[1].set_title("Chilenos vs Migrantes en Espacio PCA", fontsize=13, fontweight='bold')
axes[1].legend(markerscale=3, fontsize=10)

fig.suptitle("Fig. 19: Visualización PCA — Clustering de Perfiles Laborales (CASEN 2024)",
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig(os.path.join(FIG, 'fig19_pca_clusters.png'))
plt.show()

In [ ]:
# Figura 20: Perfil de centroides (radar/barras)
fig, ax = plt.subplots(figsize=(14, 6))

# Normalizar centroides al rango [0,1] para visualización comparada
cent_norm = df_centroides.copy()
for col in cent_norm.columns:
    cmin, cmax = cent_norm[col].min(), cent_norm[col].max()
    if cmax > cmin:
        cent_norm[col] = (cent_norm[col] - cmin) / (cmax - cmin)

x = np.arange(len(VARS_CLU))
width = 0.8 / K_FINAL

for i, (cluster, row) in enumerate(cent_norm.iterrows()):
    ax.bar(x + i*width - (K_FINAL-1)*width/2, row.values, width=width,
           label=cluster, color=colors_clust[i], alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(VARS_CLU, rotation=30, ha='right', fontsize=10)
ax.set_ylabel("Valor normalizado (0-1)")
ax.set_title("Fig. 20: Perfil Normalizado de Centroides por Cluster",
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'fig20_perfil_centroides.png'))
plt.show()

In [ ]:
# Interpretación de clusters
interp = [
    "=== INTERPRETACION DE PERFILES LABORALES (K-Means) ===",
    "",
    "Cada cluster representa un perfil diferenciado de insercion laboral.",
    "Se interpretan segun los valores de los centroides:",
    "",
    "[Inspeccionar tabla de centroides para asignar etiquetas descriptivas]",
    "",
    "Posibles perfiles tipicos:",
    "  - 'Insercion formal estable': alto ingreso, contrato, cotiza, jornada completa",
    "  - 'Insercion precaria': bajo ingreso, sin contrato, sin cotizacion",
    "  - 'Migrantes subempleados calificados': alta edu, baja remuneracion, informal",
    "  - 'Cuenta propia informal': variable, sin cotizacion, horas irregulares",
    "  - 'Alta intensidad laboral': muchas horas, formalidad media, ingresos variables",
    "",
    "Nota: La composicion migratoria por cluster muestra si los migrantes",
    "se concentran desproporcionadamente en clusters de menor calidad.",
]
for l in interp:
    print(l)

# Visualización de composición migratoria por cluster
fig, ax = plt.subplots(figsize=(10, 5))
comp_plot = cas_ocup.groupby('cluster')['migrante'].mean() * 100
tasa_global = cas_ocup['migrante'].mean() * 100

bars = ax.bar(
    [f"Cluster {k}" for k in comp_plot.index],
    comp_plot.values,
    color=['#c0392b' if v > tasa_global else '#27ae60' for v in comp_plot.values],
    edgecolor='white', alpha=0.85
)
ax.axhline(tasa_global, color='#0033A0', linestyle='--', linewidth=2,
           label=f'Tasa global: {tasa_global:.1f}%')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f"{bar.get_height():.1f}%", ha='center', fontweight='bold')
ax.set_ylabel("% de Migrantes en el Cluster")
ax.set_title("Fig. 21: Concentración de Migrantes por Cluster de Inserción Laboral",
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'fig21_migrantes_por_cluster.png'))
plt.show()

## 6. Síntesis del análisis no supervisado

In [ ]:
# Resumen final: validación de hipótesis H5 (heterogeneidad migrante)
print("=== SÍNTESIS: HETEROGENEIDAD DE INSERCIÓN MIGRANTE ===")
print()

# Distribución de migrantes por cluster
dist_mig = cas_ocup[cas_ocup['migrante']==1]['cluster'].value_counts(normalize=True)*100
dist_chil = cas_ocup[cas_ocup['migrante']==0]['cluster'].value_counts(normalize=True)*100

df_dist = pd.DataFrame({
    'Migrantes (%)': dist_mig,
    'Chilenos (%)': dist_chil
}).round(1).sort_index()
df_dist.index = [f"Cluster {i}" for i in df_dist.index]
print(df_dist.to_string())
print()
print("=== MÉTRICAS FINALES K-MEANS ===")
sil_final = silhouette_score(X_sc, cas_ocup['cluster'], sample_size=15000, random_state=42)
db_final  = davies_bouldin_score(X_sc, cas_ocup['cluster'])
print(f"Silhouette Score: {sil_final:.4f}")
print(f"Davies-Bouldin Score: {db_final:.4f}")
print(f"Inercia: {km_final.inertia_:,.0f}")
print()
print("Interpretación: Si los migrantes se concentran en clusters de menor")
print("IILM, esto evidencia segmentación estructural del mercado laboral,")
print("confirmando la H5 (heterogeneidad) y apoyando H3 (precariedad).")